# 04장 보안 실습 — 파일 조사와 경계 보존


## Goal

공백·개행 파일명을 보존하고 사본의 변경을 확인합니다.

[교안과 분석 질문](../../04-file-io/04-4-filesystem-investigation.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-04-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'provenance.txt': 'case_id=COURSE-IR-002\nsource_type=synthetic\nsource_host=lab-web-01\nsource_timezone=Asia/Seoul\nwindow_start=2026-09-10T09:00:00+09:00\nwindow_end=2026-09-10T10:00:00+09:00\ncollector=course-author\ncollection_scope=selected teaching records only\naudit_coverage=partial\nauthorization=offline classroom analysis\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: 파일 4개, 줄 수 5, 원본 평문 보존

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. 실행하지 않을 교육용 파일 만들기


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
mkdir "$COURSE_OUT/tree"
printf 'plain evidence\n' > "$COURSE_OUT/tree/file with spaces.txt"
printf 'hidden note\n' > "$COURSE_OUT/tree/.note"
printf 'option-like filename\n' > "$COURSE_OUT/tree/-n"
printf 'new line filename\n' > "$COURSE_OUT/tree/line"$'\n'"break.txt"
printf 'created_files=4\n'


### 2. NUL 구분 목록으로 파일 경계 보존


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
find "$COURSE_OUT/tree" -type f -print0 > "$COURSE_OUT/files.nul"
count=0
while IFS= read -r -d '' path; do
    test -f "$path"
    count=$((count + 1))
done < "$COURSE_OUT/files.nul"
test "$count" -eq 4
printf 'nul_records=%s\n' "$count"


### 3. 줄 수와 파일 수의 차이 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
find "$COURSE_OUT/tree" -type f -print > "$COURSE_OUT/files.lines"
test "$(wc -l < "$COURSE_OUT/files.lines")" -eq 5
printf 'line_count=5 but file_count=4\n'


### 4. 사본 변경과 원본 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
cp "$COURSE_OUT/tree/file with spaces.txt" "$COURSE_OUT/review-copy.txt"
printf 'analyst annotation\n' >> "$COURSE_OUT/review-copy.txt"
status=0
cmp -s "$COURSE_OUT/tree/file with spaces.txt" "$COURSE_OUT/review-copy.txt" || status=$?
test "$status" -eq 1
test "$(cat "$COURSE_OUT/tree/file with spaces.txt")" = 'plain evidence'
printf 'copy_changed=yes source_text_preserved=yes\n'


## Red Team ↔ Blue Team 사례 분석

### 사례: 최근 변경된 실행 가능 파일

**Red Team 질문:** 업무용 파일을 변경할 수 있는 주체와 그 파일을 실행하는 주체 사이에 권한 차이가 있는가? 파일 쓰기 가능 여부는 검토 조건 하나이며 실행 방식·경로 신뢰·추가 접근 통제를 모르고 영향을 확정할 수 없습니다.

| 연결 단계 | 분석 내용 |
|---|---|
| Goal / Boundary | 파일 변경이 의도한 데이터 수정 범위를 넘을 수 있는지 검토 |
| Command / Observation | find로 범위 제한 → stat으로 소유자·시각 → file·strings로 내용 단서 → sha256sum으로 사본 비교 |
| System Change / Artifact | 파일 내용·권한·디렉터리 항목의 변경. mtime·ctime·해시는 서로 다른 정보 |
| Log prerequisite | 기존 파일 감사·배포 기록·EDR가 있어야 변경 주체와 실행을 연결할 수 있음 |
| Blue Team Investigation | 패키지·승인 배포·이전 해시·실행 프로세스·열린 파일과 비교 |
| Detection | 기준선 변화와 비승인 변경, 후속 실행 근거를 결합. 확장자나 최근 시각만으로 확정하지 않음 |
| Mitigation | 실행 경로와 상위 디렉터리의 변경 권한 분리, 배포 통제, 사본 보존 후 복구 |

**반례와 해설:** 정기 업데이트도 최근 mtime과 다른 해시를 만듭니다. 반대로 해시가 같아도 부적절한 권한 위임은 남아 있을 수 있습니다. 문자열에서 URL을 찾았다는 사실은 해당 URL에 통신했다는 증거가 아닙니다. 분석 목적으로 의심 파일을 실행하지 않습니다.

**제출 과제:** 실습의 공백·개행 파일명 중 하나를 선택해 안전한 식별 방법, 정상 배포 가설, 확인할 실행 근거를 적습니다. 파일 목록 네 개를 줄 수 다섯 개와 혼동하지 않고, 내용 변화·권한 변화·실행을 따로 설명하면 통과입니다.


## 역할별 분석 기록

같은 실행 결과로 아래 항목을 작성하고 상대 관점에서 검토합니다. 자동 테스트는 계산과 원본 보존만 확인하며 이 서술 과제는 강사 또는 동료 검토 대상입니다.

| 항목 | 학생 작성 |
|---|---|
| Red Team 목적·필요 조건 | 관찰에서 도출한 질문과 전제 |
| 실제 관찰 | 파일·행·이벤트 ID와 출력 |
| Artifact·로깅 전제 | 확보한 자료와 필요한 기록 기능 |
| Blue Team 조사 | 정상 반례·추가 근거·수집 한계 |
| 탐지·완화 | 필요한 필드·오탐 사례·확인된 원인에 맞는 조치 |


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
